# Module 2: Units and the Memory Budget

In Module 1 you reached a server you own and traced one request. Before you open the box in Module 3, you need the units the rest of the workshop counts in. This module is short and runs anywhere, no GPU needed. You count parameters and bytes, turn a model size into gigabytes, estimate tokens, and split a card's memory into weights, the KV cache, and overhead. By the end you can size a model against a card in your head.

## Learning objectives
- Define a parameter and a byte, and read precision as bytes per number
- Turn a model's parameter count into a memory footprint at FP16, FP8, and INT4
- State what a token is in characters, and why it is the unit a server bills
- Name the two GPU numbers that matter, VRAM and memory bandwidth
- Split a card's memory into weights, KV cache, and overhead, and read the budget at FP16 versus FP8

## Prerequisites
- Finished Module 1
- No GPU and no server needed, the whole module is arithmetic
- About 6 minutes

References: [vLLM](https://docs.vllm.ai) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/design/metrics/) &middot; [Qwen3-4B model card](https://huggingface.co/Qwen/Qwen3-4B) &middot; [PagedAttention paper](https://arxiv.org/abs/2309.06180)

## Memory design basics

A model is a pile of numbers, and a GPU is a fixed amount of fast memory. Most of what you tune later is the fit between the two.

- A **parameter** is one number the model learned. A 4B model has 4 billion of them.
- **Precision** is how many bytes one parameter takes. FP16 and BF16 are 2 bytes, FP8 is 1, INT4 is half a byte.
- **Model size** is parameters times bytes. A 4B model at FP16 is about 8 GB.
- **VRAM** is how much the card holds. **Bandwidth** is how fast it reads. Those two numbers decide what fits and how fast it runs.
- The card's memory splits three ways: the weights, the KV cache, and some overhead. The weights are fixed. The KV cache is the part you trade for more users.

![The 20 GB card split into weights, overhead, and the KV cache, at FP16 versus FP8, with smaller weights leaving more room for the cache](images/02_units_and_memory_budget_architecture.png)

## 1. Setup

No server and no GPU. This module is arithmetic, so the only dependency is the plotting library for the budget chart at the end. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q matplotlib

## 2. Parameters, bytes, and precision

A parameter is a number, and precision is how many bytes you spend to store it. Fewer bytes per number is the idea behind quantization, which Module 5 takes to real models. Read the bytes per number for each precision.

In [ ]:
# Bytes per parameter at each precision. Fewer bytes is the basis of quantization (Module 5).
bytes_per_param = {"FP32": 4, "FP16": 2, "BF16": 2, "FP8": 1, "INT4": 0.5}
for name, b in bytes_per_param.items():
    print(f"{name:5}: {b} bytes per parameter")

**What you should see:** four bytes down to half a byte, depending on precision. FP16 is the common default at 2 bytes. Module 5 spends fewer and measures what it costs in accuracy.

## 3. Model size from parameters

Model size is one multiplication: parameters times bytes per parameter. This is the first number you run before you pick a card, because the weights have to fit with room left over.

In [ ]:
# Model size = parameters * bytes per parameter.
params = 4e9   # a 4B model
for prec in ("FP16", "FP8", "INT4"):
    gb = params * bytes_per_param[prec] / 1e9
    print(f"4B at {prec:4}: {gb:.1f} GB of weights")

**What you should see:** about 8 GB at FP16, 4 GB at FP8, 2 GB at INT4. The same model, a quarter of the memory, just by spending fewer bytes per weight. That freed memory becomes KV cache, which is section 6.

## 4. What a token is

A token is not a word. A tokenizer splits text into pieces of about four characters, so you can estimate token counts in your head before the tokenizer runs. Tokens are the unit a provider bills, the unit the model emits one at a time, and the unit the KV cache stores.

In [ ]:
# A token is about 4 characters of English. The tokenizer is exact; this is the head estimate.
sample = "Owning your inference means owning the layer under your agent."
est_tokens = len(sample) // 4
print(f"text: {sample!r}")
print(f"characters: {len(sample)}  ->  about {est_tokens} tokens (4 chars each)")

**What you should see:** a token estimate near a quarter of the character count. Good enough to size a context window or a bill without running the tokenizer.

## 5. VRAM and bandwidth

Two numbers describe your card. VRAM is how much fits. Bandwidth is how fast the GPU reads it. They map to the two phases you traced in Module 1: prefill is limited by compute, and decode is limited by bandwidth, because each token reloads the whole weight set out of memory. That gives a single-request speed limit you can compute now.

In [ ]:
# The two card numbers, and the decode speed limit they imply.
card_vram_gb = 20.0     # RTX 4000 Ada, from nvidia-smi
bandwidth_gbs = 360.0   # from the card datasheet
weights_gb = params * bytes_per_param["FP16"] / 1e9

# Decode reloads every weight to make one token, so one request goes no faster than bandwidth / weights.
decode_ceiling = bandwidth_gbs / weights_gb
print(f"VRAM      : {card_vram_gb:.0f} GB")
print(f"bandwidth : {bandwidth_gbs:.0f} GB/s")
print(f"single-request decode ceiling: about {decode_ceiling:.0f} tokens/s")

**What you should see:** a decode ceiling near 45 tokens per second for an 8 GB model on a 360 GB/s card. No software makes one request faster than that floor. Module 4 draws this as a plot and explains why, and Module 3 measures it on your own GPU.

## 6. The memory budget

The card's VRAM is a budget. The weights take a fixed slice, overhead takes a little, and what is left is KV cache, the part that decides how many users fit. Spend fewer bytes on the weights and the cache slice grows. Split the budget at FP16 and FP8 and see the difference.

In [ ]:
# Split a 20 GB card into weights, overhead, and KV cache, at FP16 and FP8.
overhead_gb = 1.0
util = 0.90   # --gpu-memory-utilization: the share vLLM may claim

for prec in ("FP16", "FP8"):
    w = params * bytes_per_param[prec] / 1e9
    kv = util * card_vram_gb - w - overhead_gb
    print(f"{prec:4}: weights {w:4.1f} GB + overhead {overhead_gb:.1f} GB -> KV cache {kv:4.1f} GB")

**What you should see:** at FP16 the KV cache slice is about 9 GB, and at FP8 it is about 13 GB. Same card, more room for users, from spending fewer bytes per weight. Module 3 turns that KV slice into an exact per-token cost and a concurrent-request ceiling. Module 5 shrinks the weights for real.

In [ ]:
# A stacked bar of the budget, FP16 versus FP8.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

labels = ["FP16", "FP8"]
weights = [params * bytes_per_param[p] / 1e9 for p in labels]
over = [overhead_gb, overhead_gb]
kv = [util * card_vram_gb - w - overhead_gb for w in weights]

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(labels, weights, label="weights")
ax.bar(labels, over, bottom=weights, label="overhead")
ax.bar(labels, kv, bottom=[w + o for w, o in zip(weights, over)], label="KV cache")
ax.set_ylabel("GB of the 20 GB card")
ax.set_title("Smaller weights leave more room for the KV cache")
ax.legend()
fig.tight_layout()

**What you should see:** two bars, with the KV cache slice clearly taller under FP8. This is the picture Module 3's concurrency ceiling and the quantization module both build on.

## Things to know

- **Model size is the first number you run.** Parameters times bytes. The cache, the concurrency, and the cost all start from how much the weights take.
- **Fewer bytes per weight buys concurrency.** Quantization does not only shrink the model, it frees memory that becomes KV cache, so more users fit on the same card. Module 5 measures that.
- **Decode has a floor.** One request goes no faster than bandwidth over weight size, near 45 tokens per second here. Batching is how you serve many users at once without each one going faster, which Modules 7 and 8 turn into an operating point.
- **A token is about four characters.** Estimate in your head, confirm with the tokenizer when it matters.

## Try it yourself

**Size a bigger model.** Change `your_params` to 8e9, then 70e9, and read the weights at each precision. Find where the model stops fitting your 20 GB card. **Stretch:** find the largest model that fits at INT4 with at least 4 GB left for the KV cache.

**Move the budget.** Raise `util` toward 0.95 and watch the KV cache slice grow. That single flag is one of the levers Module 8 can test in the tuning loop.

In [ ]:
# Change these, then run the cell.
your_params = 8e9          # try 8e9, 70e9
your_precision = "FP16"    # try FP8, INT4

w = your_params * bytes_per_param[your_precision] / 1e9
kv = util * card_vram_gb - w - overhead_gb
fits = "fits" if kv > 0 else "does NOT fit"
print(f"{your_params/1e9:.0f}B at {your_precision}: weights {w:.1f} GB, KV cache {kv:.1f} GB -> {fits}")

## Summary

- A parameter is a number, and precision is its size in bytes. FP16 is 2 bytes, FP8 is 1, INT4 is half.
- Model size is parameters times bytes. A 4B model is about 8 GB at FP16.
- A token is about four characters, the unit billed, emitted, and cached.
- The card's VRAM splits into weights, overhead, and KV cache. Smaller weights leave more room for the cache and more users.

## Next

**Module 3: Prefill, Decode, and the KV Cache.** You can size a model now. Next you build attention and the KV cache by hand, watch generation go from quadratic to linear, and measure the exact per-token cache cost on your own GPU.